# Funções geradoras de alias

Em APIs REST é extremamente comum usar camelCase nos dados (de/para JSON) e snake_case nos nomes dos campos.

Uma abordagem é usar as definições de alias da aula anterior.

Mas fazer isso para centenas de campos e dezenas de modelos é tedioso e sujeito a erro.

Converter snake_case em camelCase é bem sistemático, e poderíamos escrever uma função para isso.

Na verdade, há bibliotecas de terceiros que fazem exatamente isso.

O Pydantic também faz.

Essas funções de conversão estão documentadas [aqui](https://docs.pydantic.dev/latest/api/config/#pydantic.alias_generators).

Vejamos:

In [1]:
from pydantic.alias_generators import to_camel, to_snake, to_pascal

São funções comuns:

In [2]:
print(f'to_camel("last_name"): {to_camel("last_name")!r}')

to_camel("last_name"): 'lastName'


In [3]:
print(f'to_snake("lastName"): {to_snake("lastName")!r}')

to_snake("lastName"): 'last_name'


In [4]:
print(f'to_pascal("last_name"): {to_pascal("last_name")!r}')

to_pascal("last_name"): 'LastName'


Elas existem porque podemos configurar o **modelo** para gerar aliases automaticamente usando uma delas.

Mas qualquer função que converta uma string (o nome do campo) em outra (o alias) também serve.

Vejamos a geração automática de aliases que simplesmente coloca o nome em maiúsculas.

In [5]:
def make_upper(in_str: str) -> str:
    return in_str.upper()

Só uma função comum:

In [6]:
print(f'make_upper("last_name"): {make_upper("last_name")!r}')

make_upper("last_name"): 'LAST_NAME'


Agora anexamos a função à definição do modelo, claro, via `model_config`:

In [7]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class Person(BaseModel):
    model_config = ConfigDict(alias_generator=make_upper)

    id_: int
    first_name: str | None = None
    last_name: str
    age: int | None = None

Inspecionando os campos:

In [8]:
print(f"Person.model_fields: {Person.model_fields!r}")

Person.model_fields: {'id_': FieldInfo(annotation=int, required=True, alias='ID_', alias_priority=1), 'first_name': FieldInfo(annotation=Union[str, NoneType], required=False, default=None, alias='FIRST_NAME', alias_priority=1), 'last_name': FieldInfo(annotation=str, required=True, alias='LAST_NAME', alias_priority=1), 'age': FieldInfo(annotation=Union[int, NoneType], required=False, default=None, alias='AGE', alias_priority=1)}


Os aliases foram gerados automaticamente. Testando:

In [9]:
p = Person(ID_=1, LAST_NAME="Fourier", AGE=62)
print(f"p: {p!r}")

p: Person(id_=1, first_name=None, last_name='Fourier', age=62)


In [10]:
print(f"p.model_dump(): {p.model_dump()!r}")

p.model_dump(): {'id_': 1, 'first_name': None, 'last_name': 'Fourier', 'age': 62}


In [11]:
print(f"p.model_dump(by_alias=True): {p.model_dump(by_alias=True)!r}")

p.model_dump(by_alias=True): {'ID_': 1, 'FIRST_NAME': None, 'LAST_NAME': 'Fourier', 'AGE': 62}


O alias de `id_` virou `ID_` — talvez não seja o que queremos. Nesse caso, sobrescrevemos:

In [12]:
class Person(BaseModel):
    model_config = ConfigDict(alias_generator=make_upper)

    id_: int = Field(alias="ID")
    first_name: str | None = None
    last_name: str
    age: int | None = None

In [13]:
print(f"Person.model_fields: {Person.model_fields!r}")

Person.model_fields: {'id_': FieldInfo(annotation=int, required=True, alias='ID', alias_priority=2), 'first_name': FieldInfo(annotation=Union[str, NoneType], required=False, default=None, alias='FIRST_NAME', alias_priority=1), 'last_name': FieldInfo(annotation=str, required=True, alias='LAST_NAME', alias_priority=1), 'age': FieldInfo(annotation=Union[int, NoneType], required=False, default=None, alias='AGE', alias_priority=1)}


In [14]:
p = Person(ID=1, LAST_NAME="Fourier", AGE=62)
print(f"p: {p!r}")

p: Person(id_=1, first_name=None, last_name='Fourier', age=62)


In [15]:
print(f"p.model_dump(): {p.model_dump()!r}")

p.model_dump(): {'id_': 1, 'first_name': None, 'last_name': 'Fourier', 'age': 62}


In [16]:
print(f"p.model_dump(by_alias=True): {p.model_dump(by_alias=True)!r}")

p.model_dump(by_alias=True): {'ID': 1, 'FIRST_NAME': None, 'LAST_NAME': 'Fourier', 'AGE': 62}


Normalmente queremos só que os aliases sejam a versão camelCase dos nomes — então usamos a função utilitária do Pydantic `to_camel`:

In [17]:
class Person(BaseModel):
    model_config = ConfigDict(alias_generator=to_camel)

    id_: int = Field(alias="id")
    first_name: str | None = None
    last_name: str
    age: int | None = None

Inspecionando os campos:

In [18]:
print(f"Person.model_fields: {Person.model_fields!r}")

Person.model_fields: {'id_': FieldInfo(annotation=int, required=True, alias='id', alias_priority=2), 'first_name': FieldInfo(annotation=Union[str, NoneType], required=False, default=None, alias='firstName', alias_priority=1), 'last_name': FieldInfo(annotation=str, required=True, alias='lastName', alias_priority=1), 'age': FieldInfo(annotation=Union[int, NoneType], required=False, default=None, alias='age', alias_priority=1)}


Temos os aliases desejados, que usamos da forma habitual:

In [19]:
p = Person(id=1, lastName="Fourier", age=62)
print(f"p: {p!r}")

p: Person(id_=1, first_name=None, last_name='Fourier', age=62)


In [20]:
print(f"p.model_dump(): {p.model_dump()!r}")

p.model_dump(): {'id_': 1, 'first_name': None, 'last_name': 'Fourier', 'age': 62}


In [21]:
print(f"p.model_dump(by_alias=True): {p.model_dump(by_alias=True)!r}")

p.model_dump(by_alias=True): {'id': 1, 'firstName': None, 'lastName': 'Fourier', 'age': 62}


Quando os dados trazem palavras reservadas do Python, é costume nomear o campo com um `_` no final, por exemplo:
- `id` -> `id_`
- `list` -> `list_`
- `filter` -> `filter_`

(`id`, `list` e `filter` são, a rigor, nomes de *built-ins*, não palavras reservadas — mas sobrescrevê-los como atributos de classe
confunde e é evitado pela mesma razão.)

Poderíamos usar o método que acabamos de ver, com um gerador de alias, sobrescrevendo esses casos um a um:

In [22]:
class Model(BaseModel):
    model_config = ConfigDict(alias_generator=to_camel)

    id_: int = Field(alias="id")
    list_: list[str] = Field(alias="list")
    filter_: dict = Field(alias="filter")
    number_elements: list[int]

Assim ficamos com os seguintes aliases:

In [23]:
print(f"Model.model_fields: {Model.model_fields!r}")

Model.model_fields: {'id_': FieldInfo(annotation=int, required=True, alias='id', alias_priority=2), 'list_': FieldInfo(annotation=list[str], required=True, alias='list', alias_priority=2), 'filter_': FieldInfo(annotation=dict, required=True, alias='filter', alias_priority=2), 'number_elements': FieldInfo(annotation=list[int], required=True, alias='numberElements', alias_priority=1)}


Mas, como geramos esses aliases de forma totalmente consistente (converter para camelCase e retirar o `_` final), dá para fazer tudo com
uma função geradora customizada:

In [24]:
def make_alias(field_name: str) -> str:
    alias = to_camel(field_name)
    return alias.removesuffix("_")

In [25]:
print(f'make_alias("id_"): {make_alias("id_")!r}')

make_alias("id_"): 'id'


In [26]:
print(f'make_alias("number_elements"): {make_alias("number_elements")!r}')

make_alias("number_elements"): 'numberElements'


E usamos essa função no modelo:

In [27]:
class Model(BaseModel):
    model_config = ConfigDict(alias_generator=make_alias)

    id_: int
    list_: list[str]
    filter_: dict
    number_elements: list[int]

In [28]:
print(f"Model.model_fields: {Model.model_fields!r}")

Model.model_fields: {'id_': FieldInfo(annotation=int, required=True, alias='id', alias_priority=1), 'list_': FieldInfo(annotation=list[str], required=True, alias='list', alias_priority=1), 'filter_': FieldInfo(annotation=dict, required=True, alias='filter', alias_priority=1), 'number_elements': FieldInfo(annotation=list[int], required=True, alias='numberElements', alias_priority=1)}
